In [1]:
from collections import Counter

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import poisson

In [2]:
columns = [
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament",
    "neutral",
]

data = pd.read_csv("../data/results.csv")
data = data[data["date"] >= "2023-01-01"][columns]

teams_to_evaluate = {"Brazil"}
teams_to_consider = set()
while len(teams_to_evaluate) > 0:
    team = teams_to_evaluate.pop()
    teams_to_consider.add(team)

    matches = data[(data["home_team"] == team) | (data["away_team"] == team)]

    unique_opponents = set(matches["home_team"].unique())
    unique_opponents.update(matches["away_team"].unique())

    unique_opponents.discard(team)
    if len(unique_opponents) <= 5:
        teams_to_consider.discard(team)
        continue

    teams_to_evaluate.update(unique_opponents)

    teams_to_evaluate = teams_to_evaluate - teams_to_consider

data = data[
    (data["home_team"].isin(teams_to_consider))
    & (data["away_team"].isin(teams_to_consider))
]

data

,date,home_team,away_team,home_score,away_score,tournament,neutral
45770,2023-01-02,Philippines,Indonesia,1,2,AFF Championship,False
45771,2023-01-02,Thailand,Cambodia,3,1,AFF Championship,False
45772,2023-01-03,Vietnam,Myanmar,3,0,AFF Championship,False
45773,2023-01-03,Malaysia,Singapore,4,1,AFF Championship,False
45774,2023-01-06,Iraq,Oman,0,0,Gulf Cup,False
...,...,...,...,...,...,...,...
49066,2026-01-18,Grenada,Jamaica,0,1,Friendly,False
49067,2026-01-18,Morocco,Senegal,0,1,African Cup of Nations,False
49068,2026-01-22,Panama,Mexico,0,1,Friendly,False
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,False


In [3]:
home_games = (
    data.groupby("home_team")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="home_games")
)

away_games = (
    data.groupby("away_team")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="away_games")
)

games = away_games.merge(
    home_games, left_on="away_team", right_on="home_team", how="outer"
)

games["home_games"] = games["home_games"].fillna(0).astype(int)
games["away_games"] = games["away_games"].fillna(0).astype(int)
games["total_games"] = games["home_games"] + games["away_games"]

games = games.sort_values(by="total_games", ascending=False, ignore_index=True)

games["team"] = np.where(
    games["home_team"].isna(), games["away_team"], games["home_team"]
)

games = games[["team", "total_games"]]
games = games[games["total_games"] >= 20]
teams = games["team"].tolist()

data = data[(data["home_team"].isin(teams)) & (data["away_team"].isin(teams))]

data["home_idx"] = data["home_team"].apply(lambda x: teams.index(x))
data["away_idx"] = data["away_team"].apply(lambda x: teams.index(x))

print(f"Number of teams: {len(teams)}")
data

Number of teams: 180


,date,home_team,away_team,home_score,away_score,tournament,neutral,home_idx,away_idx
45770,2023-01-02,Philippines,Indonesia,1,2,AFF Championship,False,97,20
45771,2023-01-02,Thailand,Cambodia,3,1,AFF Championship,False,7,173
45772,2023-01-03,Vietnam,Myanmar,3,0,AFF Championship,False,33,138
45773,2023-01-03,Malaysia,Singapore,4,1,AFF Championship,False,39,96
45774,2023-01-06,Iraq,Oman,0,0,Gulf Cup,False,15,6
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Grenada,Jamaica,0,1,Friendly,False,146,2
49067,2026-01-18,Morocco,Senegal,0,1,African Cup of Nations,False,10,26
49068,2026-01-22,Panama,Mexico,0,1,Friendly,False,4,1
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,False,37,1


In [4]:
def log_likelihood_negative(strengths, data):
    inner_data = data.copy()
    strengths = np.array(strengths[:-1])
    home_strength = strengths[-1]

    inner_data["home_strength"] = strengths[inner_data["home_idx"]]
    inner_data["home_strength"] += home_strength * (inner_data["neutral"] == 0)
    inner_data["away_strength"] = strengths[inner_data["away_idx"]]

    inner_data["likelihood"] = poisson.logpmf(
        inner_data["home_score"], inner_data["home_strength"]
    ) + poisson.logpmf(inner_data["away_score"], inner_data["away_strength"])

    return -float(inner_data["likelihood"].sum())


log_likelihood_negative(np.ones(len(teams) + 1), data)

8683.884273376601

In [5]:
res = minimize(
    fun=lambda x: log_likelihood_negative(x, data),
    x0=np.ones(len(teams) + 1),
    method="L-BFGS-B",
    bounds=[(1e-6, None)] * (len(teams) + 1),
    options={"maxiter": 1000},
)

res.success

True

In [6]:
games["idx"] = games["team"].apply(lambda x: teams.index(x))
games["strength"] = res.x[games["idx"]]
games.drop(columns=["idx"], inplace=True)
games.sort_values(by="strength", ascending=False, ignore_index=True, inplace=True)

games

,team,total_games,strength
0,Japan,35,2.945638
1,Spain,37,2.564932
2,Norway,30,2.482379
3,Russia,20,2.446378
4,Portugal,36,2.430014
...,...,...,...
175,Antigua and Barbuda,21,0.306309
176,San Marino,30,0.299697
177,Andorra,30,0.212886
178,Liechtenstein,30,0.187850


In [7]:
home = "Brazil"
away = "Croatia"

home_strength = res.x[teams.index(home)]
away_strength = res.x[teams.index(away)]

n = 1_000_000
home_goals = np.random.poisson(home_strength, n)
away_goals = np.random.poisson(away_strength, n)

home_wins = (home_goals > away_goals).sum() / n
away_wins = (home_goals < away_goals).sum() / n
draws = (home_goals == away_goals).sum() / n

most_common_score = Counter(zip(home_goals, away_goals, strict=False)).most_common(1)[
    0
][0]

home_goals, away_goals = most_common_score

print(f"{home} strength: {home_strength:.2f}")
print(f"{away} strength: {away_strength:.2f}")

print()
print(f"{home} wins: {home_wins:.2%}")
print(f"Draws: {draws:.2%}")
print(f"{away} wins: {away_wins:.2%}")

print()
print(f"Most common score: {home} {home_goals} - {away_goals} {away}")

Brazil strength: 1.52
Croatia strength: 1.80

Brazil wins: 32.88%
Draws: 22.71%
Croatia wins: 44.40%

Most common score: Brazil 1 - 1 Croatia
